### Оглавление ноутбука

- 🛠 Catboost Classifier и его параметры
- 📂 Настраиваем обучение по фолдам
- 🔧 Best practices по настройке Catboost
- 🔋 "Батарейки в комплекте!"
- 🤷‍ Другие параметры и режимы
- 🧸 Выводы и заключения


Что такое CatBoost?

**CatBoost** — это open-source библиотека градиентного бустинга на решающих деревьях с поддержкой категориальных фичей из коробки, преемник алгоритма MatrixNet, разработанного Яндексом.

В плане простоты использования и легкости входа для новичков, пожалуй, является топ-1 библиотекой для табличных данных и вот почему:

- **Принимает категориальные фичи** сразу без всякой предварительной обработки
- Чтобы перенести обучение с **CPU** на **GPU** достаточно поменять значение 1 параметра
- Даже с дефолтными параметрами выдает хорошую точность модели
- Основные параметры не константные, а **подбираются самой библиотекой**, в зависимости от размера входных данных
- Может принимать текстовые признаки, эмбеддинги, временные признаки
- Без дополнительных манипуляций встраивается в стандартные пайплайны (например, sklearn)
- Идет в комплекте с "батарейками": feature_selection, object_selection, cross_validation, grid_search и пр.


Минусы (почти нет):

- Не умеет обрабатывать пропуски в данных (нужно самим избавляться или заполнять NaN перед подачей в модель)
- Не все метрики и лоссы доступны при обучении на GPU
- Не умеет делать кофе ☕


Рекомендуемый workflow:

В общем и целом, рекомендуем начинать всегда именно с этой библиотеки, особенно если только начинаете вкатываться в соревнования.

**Алгоритм примерно следующий:**

1. Скачали данные
2. Провели быстрый EDA
3. Собрали список кат-фичей, закинули данные в CatBoost
4. Посмотрели, что модель дает с дефолтными фичами и параметрами (baseline)
5. Отправили сабмишен на лидерборд
6. Начинаем настройку валидации, feature engineering, тюнинг модели, прочие манипуляции и танцы с бубном


## Задание 1: Установка библиотек и импорты

Установи необходимые библиотеки (catboost, seaborn) и импортируй numpy и pandas.


In [ ]:
# Твой код здесь
# !pip install seaborn catboost -q

# import numpy as np
# import pandas as pd


## Задание 2: Загрузка данных

Загрузи train датасет из файла `"../data/quickstart_train.csv"`.

Выведи:
- Размер датасета (shape)
- Гистограммы признаков (можно использовать figsize=(25, 6), layout=(-1, 6))
- 3 случайных примера из датасета


In [ ]:
# Твой код здесь
path = "../data/quickstart_train.csv"

# train = pd.read_csv(...)
# print("train", train.shape)
# train.hist(...)
# train.sample(...)


## Задание 3: Группировка и подготовка признаков

Определи следующие списки:
- `cat_features` = ["model", "car_type", "fuel_type"] — категориальные признаки
- `targets` = ["target_class", "target_reg"] — целевые переменные  
- `features2drop` = ["car_id"] — признаки для удаления

Затем:
1. Создай `filtered_features` — список всех признаков, исключая targets и features2drop
2. Создай `num_features` — список числовых признаков (все из filtered_features, кроме категориальных)
3. Выведи списки cat_features, количество num_features и targets
4. Преобразуй все категориальные столбцы в строковый тип (чтобы избавиться от NaN)


In [ ]:
# Твой код здесь
cat_features = ...
targets = ...
features2drop = ...

# filtered_features = [i for i in train.columns if ...]
# num_features = [i for i in filtered_features if ...]

# print("cat_features", cat_features)
# print("num_features", len(num_features))
# print("targets", targets)

# for c in cat_features:
#     train[c] = train[c].astype(str)


# CatBoost Classifier и его параметры

Задача - использовать CatBoost для классификации поломок. Посмотрим, сможет ли алгоритм справиться с поставленной задачей.

Перед использованием рассмотрим параметры модели ([подробнее тут](https://catboost.ai/en/docs/references/training-parameters/common)):


**Базовые параметры:**

- `iterations` (синонимы: `num_boost_round`, `n_estimators`, `num_trees`) - максимальное количество деревьев (по умолчанию 1000)
- `learning_rate` или `eta` – скорость обучения (по умолчанию ≈ 0.03)
- `depth` (или `max_depth`) - глубина дерева (по умолчанию 6, максимум 16)
- `cat_features` - список категориальных признаков

**Режим обучения:**

- `loss_function` или `objective` - функция потерь (logloss для классификации, RMSE для регрессии)
- `eval_metric` - валидационная метрика для обнаружения переобучения
- `custom_metric` - дополнительные отслеживаемые метрики
- `early_stopping_rounds` - число итераций для ранней остановки
- `use_best_model` - если True, вернется модель с лучшей метрикой на валидации

**Регуляризация:**

- `l2_leaf_reg` (или `reg_lambda`) – коэффициент L2 регуляризации (по умолчанию 3.0)
- `min_data_in_leaf` - минимальное количество сэмплов в листе
- `max_leaves` - максимальное количество листьев
- `subsample` - доля выборки для каждого дерева
- `colsample_bylevel` - доля признаков на каждом сплите

**Полезные параметры:**

- `random_seed` или `random_state` – для воспроизводимости
- `task_type` - CPU или GPU
- `thread_count` - число потоков (по умолчанию -1 = все ядра)
- `verbose` - вывод информации


## Задание 4: Импорт CatBoost

Импортируй из библиотеки catboost:
- CatBoostClassifier
- CatBoostRegressor  
- Pool


In [ ]:
# Твой код здесь
# from catboost import ...


## Задание 5: Разделение данных на train и test

Импортируй `train_test_split` из `sklearn.model_selection`.

Создай X и y:
- X — все признаки из `filtered_features` (исключая targets)
- y — целевая переменная "target_class"

Раздели данные с параметрами: test_size=0.2, random_state=42


In [ ]:
# Твой код здесь
# from sklearn.model_selection import train_test_split

# X = train[filtered_features].drop(targets, axis=1, errors="ignore")
# y = train["target_class"]

# X_train, X_test, y_train, y_test = train_test_split(...)


## Задание 6: Обучение первой модели CatBoost

Создай и обучи модель CatBoostClassifier с параметрами по умолчанию.

Параметры модели:
- `thread_count=-1`
- `random_seed=42`  
- `cat_features=cat_features`

Параметры fit():
- `eval_set=(X_test, y_test)`
- `verbose=200`
- `use_best_model=True`
- `plot=False`
- `early_stopping_rounds=100`

**Обрати внимание:** CatBoost сам подберет learning_rate!


In [ ]:
# Твой код здесь
# clf = CatBoostClassifier(
#     thread_count=...,
#     random_seed=...,
#     cat_features=...
# )

# clf.fit(
#     X_train,
#     y_train,
#     eval_set=...,
#     verbose=...,
#     use_best_model=...,
#     plot=...,
#     early_stopping_rounds=...
# )


## Задание 7: Эксперимент с количеством деревьев

Обучи новую модель с параметром `iterations=100` (остальные параметры те же).

Посмотри, как изменится автоматически подобранный `learning_rate`!


In [ ]:
# Твой код здесь
# clf = CatBoostClassifier(
#     iterations=...,
#     thread_count=-1,
#     random_seed=42,
#     cat_features=cat_features
# )

# clf.fit(...)


## Задание 8: Feature Importance

Используй метод `get_feature_importance(prettified=True)` чтобы посмотреть на важность признаков в красивом виде.


In [ ]:
# Твой код здесь
# clf.get_feature_importance(...)


# Настраиваем обучение по фолдам

Кросс-валидация - это важный инструмент для оценки качества модели и борьбы с переобучением.


## Задание 9: Кросс-валидация с KFold

Настрой обучение модели с кросс-валидацией:

1. Импортируй KFold из sklearn.model_selection
2. Создай списки `clfs = []` и `scores = []`
3. Подготовь X и y на всех данных train (не разделенных)
4. Создай объект KFold с n_splits=3, shuffle=True, random_state=7575
5. В цикле по фолдам:
   - Раздели данные по индексам фолда
   - Создай Pool для train_dataset и eval_dataset (с cat_features)
   - Обучи модель CatBoostClassifier
   - Сохрани модель в список clfs
   - Получи предсказания и посчитай Recall (из sklearn.metrics)
   - Сохрани метрику в scores
6. Выведи среднее значение Recall по всем фолдам

**Подсказка:** Pool ускоряет обучение CatBoost и автоматически обрабатывает категориальные признаки.


In [ ]:
# Твой код здесь
# from sklearn.model_selection import KFold
# from sklearn.metrics import recall_score

# n_splits = 3
# clfs = []
# scores = []

# X = train[filtered_features].drop(targets, axis=1, errors="ignore")
# y = train["target_class"]

# kf = KFold(n_splits=..., shuffle=..., random_state=...)

# for train_index, test_index in kf.split(X):
#     X_train, X_test = X.iloc[train_index], X.iloc[test_index]
#     y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
#     train_dataset = Pool(data=..., label=..., cat_features=...)
#     eval_dataset = Pool(data=..., label=..., cat_features=...)
    
#     clf = CatBoostClassifier(...)
#     clf.fit(train_dataset, eval_set=eval_dataset, ...)
    
#     clfs.append(clf)
    
#     y_pred = clf.predict(...)
#     score = recall_score(...)
#     scores.append(score)

# print(f'mean Recall score ---------> {np.mean(scores):.3f}')


## Задание 10: Работа с тестовым датасетом и submission

1. Загрузи test датасет из `"../data/quickstart_test.csv"`
2. Примени те же преобразования категориальных признаков
3. Создай X_test из нужных признаков
4. Сделай предсказания с помощью одной из обученных моделей
5. Создай submission файл с результатами и сохрани его


In [ ]:
# Твой код здесь
# test = pd.read_csv("../data/quickstart_test.csv")

# for c in cat_features:
#     test[c] = test[c].astype(str)

# X_test = test[filtered_features]

# predictions = clf.predict(...)

# submission = pd.DataFrame({
#     'id': test['car_id'],
#     'target': predictions
# })
# submission.to_csv('submission.csv', index=False)


# 🔋 "Батарейки в комплекте!"

CatBoost идет с множеством полезных встроенных инструментов.


## Задание 11: Подбор гиперпараметров с GridSearchCV

Используй GridSearchCV для подбора оптимальных параметров:

1. Импортируй GridSearchCV из sklearn.model_selection
2. Определи сетку параметров (например: learning_rate, depth, l2_leaf_reg)
3. Создай GridSearchCV с моделью CatBoostClassifier
4. Обучи и выведи лучшие параметры и лучший score

**Внимание:** GridSearch может работать долго! Используй небольшие сетки параметров для экспериментов.


In [ ]:
# Твой код здесь
# from sklearn.model_selection import GridSearchCV

# param_grid = {
#     'learning_rate': ...,
#     'depth': ...,
#     'l2_leaf_reg': ...
# }

# clf = CatBoostClassifier(
#     iterations=...,
#     random_seed=42,
#     cat_features=...,
#     verbose=False
# )

# grid_search = GridSearchCV(clf, param_grid, cv=3, scoring='recall')
# grid_search.fit(X_train, y_train)

# print('Best params:', grid_search.best_params_)
# print('Best score:', grid_search.best_score_)


## Задание 12: Сохранение и загрузка модели

1. Сохрани обученную модель с помощью метода `save_model('catboost_model.cbm')`
2. Загрузи модель обратно используя `CatBoostClassifier().load_model(...)`
3. Проверь параметры загруженной модели с помощью `get_all_params()`


In [ ]:
# Твой код здесь
# Сохранение
# clf.save_model('catboost_model.cbm')

# Загрузка
# loaded_model = CatBoostClassifier()
# loaded_model.load_model('catboost_model.cbm')

# Проверка
# loaded_model.get_all_params()


## Полезные источники:

- [Официальная документация CatBoost](https://catboost.ai/)
- [CatBoost Tutorial](https://catboost.ai/en/docs/concepts/python-quickstart)
- [Параметры обучения](https://catboost.ai/en/docs/references/training-parameters/common)
- [Примеры использования](https://github.com/catboost/catboost/tree/master/catboost/tutorials)

